# 1.2 Problems as search: practice notebook

Turn a goal into a search problem: states, actions, a start and a goal test. Learn what a node, the frontier and the explored set are, and the four properties used to judge every search algorithm.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### The successor function of the water jug problem

In [ ]:
# The water jug problem from the slides: a 3 gallon jug and a 4 gallon jug.
# A state is a pair (gal3, gal4). The successor function lists every state one action away.
def successors(state):
    gal3, gal4 = state
    result = []
    result.append(("Fill-3", (3, gal4)))
    result.append(("Fill-4", (gal3, 4)))
    result.append(("Empty-3", (0, gal4)))
    result.append(("Empty-4", (gal3, 0)))
    pour = min(gal3, 4 - gal4)
    result.append(("Pour-3-into-4", (gal3 - pour, gal4 + pour)))
    pour = min(gal4, 3 - gal3)
    result.append(("Pour-4-into-3", (gal3 + pour, gal4 - pour)))
    return result

for action, new_state in successors((0, 0)):
    print(action, "->", new_state)

### Search nodes and rebuilding a path

In [ ]:
# A search node remembers its state, its parent node, the action that made it, and the path cost g.
class Node:
    def __init__(self, state, parent=None, action=None, cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = cost

    def path(self):
        # follow the parent links back to the start, then reverse
        states = []
        node = self
        while node is not None:
            states.append(node.state)
            node = node.parent
        states.reverse()
        return states

start = Node("Arad")
n1 = Node("Sibiu", start, "go Sibiu", 140)
n2 = Node("Fagaras", n1, "go Fagaras", 140 + 99)
n3 = Node("Bucharest", n2, "go Bucharest", 140 + 99 + 211)
print("path:", n3.path())
print("cost:", n3.path_cost)

## Your turn

### Task 1: Successors of a water jug state

Write `jug_successors(gal3, gal4)` that returns the sorted list of distinct states (as [gal3, gal4] lists) one action away, for a 3 gallon and a 4 gallon jug. Leave out the state itself if an action does not change it.

Hint: List all six actions, then remove repeats and the unchanged state.

In [ ]:
def jug_successors(gal3, gal4):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('jug_successors(0, 0)', [[0, 4], [3, 0]]), ('jug_successors(3, 0)', [[0, 0], [0, 3], [3, 4]]), ('jug_successors(2, 4)', [[0, 4], [2, 0], [3, 3], [3, 4]])])

### Task 2: Cost of a path on the Romania map

Write `path_cost(path, roads)`. roads is a dict like {"Arad-Sibiu": 140, ...} with each road written once in any direction. Return the total cost, or -1 if some step has no road.

Hint: Look up each pair in both directions.

In [ ]:
def path_cost(path, roads):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("path_cost(['Arad', 'Sibiu', 'Fagaras', 'Bucharest'], {'Arad-Sibiu': 140, 'Sibiu-Fagaras': 99, 'Bucharest-Fagaras': 211})", 450), ("path_cost(['Arad', 'Bucharest'], {'Arad-Sibiu': 140})", -1), ("path_cost(['Arad'], {})", 0)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Successors of a water jug state

This is the successor function S(x) of the slides for the jug problem.

In [ ]:
def jug_successors(gal3, gal4):
    states = []
    states.append([3, gal4])
    states.append([gal3, 4])
    states.append([0, gal4])
    states.append([gal3, 0])
    pour = min(gal3, 4 - gal4)
    states.append([gal3 - pour, gal4 + pour])
    pour = min(gal4, 3 - gal3)
    states.append([gal3 + pour, gal4 - pour])
    result = []
    for s in states:
        if s != [gal3, gal4] and s not in result:
            result.append(s)
    result.sort()
    return result

In [ ]:
check([('jug_successors(0, 0)', [[0, 4], [3, 0]]), ('jug_successors(3, 0)', [[0, 0], [0, 3], [3, 4]]), ('jug_successors(2, 4)', [[0, 4], [2, 0], [3, 3], [3, 4]])])

### Solution to task 2: Cost of a path on the Romania map

This is g(n), the path cost of the slides.

In [ ]:
def path_cost(path, roads):
    total = 0
    for i in range(len(path) - 1):
        a = path[i]
        b = path[i + 1]
        if a + "-" + b in roads:
            total = total + roads[a + "-" + b]
        elif b + "-" + a in roads:
            total = total + roads[b + "-" + a]
        else:
            return -1
    return total

In [ ]:
check([("path_cost(['Arad', 'Sibiu', 'Fagaras', 'Bucharest'], {'Arad-Sibiu': 140, 'Sibiu-Fagaras': 99, 'Bucharest-Fagaras': 211})", 450), ("path_cost(['Arad', 'Bucharest'], {'Arad-Sibiu': 140})", -1), ("path_cost(['Arad'], {})", 0)])